**NOTEBOOK:** 01_carga_y_limpieza.ipynb

**OBJETIVO:** Cargar, explorar y limpiar ENAHO 2025

PASO 1 : CARGAR LIBRERIAS 

In [10]:
import gdown as gd
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import IsolationForest
import warnings
warnings.filterwarnings('ignore')

print("\nSe han importando las librerias correctamente. (Recuerda estar en el entorno virtual ven(3.11.0))")



Se han importando las librerias correctamente. (Recuerda estar en el entorno virtual ven(3.11.0))


Opciones de visualizacion

In [25]:
# Configurar pandas para ver mejor los datos
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print("\nOpciones de visualización configuradas")


Opciones de visualización configuradas


PASO 2 : CARGAR DATOS

Descargar datos en local

In [12]:
FILE_ID = "16IzaVtYm6K8IJNsPjk1Aev8z0i7uIlEF"

# Crear la URL de la carpeta
folder_url = f"https://drive.google.com/drive/folders/{FILE_ID}"

# Descargar toda la carpeta
gd.download_folder(url=folder_url, quiet=False, use_cookies=False)

print("Archivos descargados satisfactoriamente!")

Retrieving folder contents


Processing file 1wA0ipdtmXvhUr5pI52lVuAc_uYRilR8V modulo_01_vivienda.csv
Processing file 1UOjZoWy8Dfc_pHI2Tdztk1DIaZXABmaE modulo_02_miembros.csv
Processing file 1SSkRbob42ZS16rDJetYCfpCkC4T6fqFY modulo_03_educacion.csv
Processing file 1snInGhsAQpMtt-yJY9aHJ-nCvY75sVRV modulo_05_empleo.csv
Processing file 1oc6-5OMJQzPUiATV0ZBJomOKc6tlX4ck modulo_11_servicios.csv
Processing file 1Cpypgeyw_XWgzXtbgrJZ5J0awXrg8NoJ modulo_16_equipamiento.csv
Processing file 1QOt6xL_c_JdhExOvkfCGXOOsSQHs9UHB modulo_34_sumarias.csv
Processing file 1JNJY-5JRd-EQNA6BEHocruOo5fOkCkQg modulo_37_programas.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1wA0ipdtmXvhUr5pI52lVuAc_uYRilR8V
To: c:\Users\diana\Documents\AI-TF\analisis\notebooks\enaho2025\modulo_01_vivienda.csv
100%|██████████| 36.2M/36.2M [00:58<00:00, 617kB/s] 
Downloading...
From: https://drive.google.com/uc?id=1UOjZoWy8Dfc_pHI2Tdztk1DIaZXABmaE
To: c:\Users\diana\Documents\AI-TF\analisis\notebooks\enaho2025\modulo_02_miembros.csv
100%|██████████| 15.7M/15.7M [00:01<00:00, 8.13MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1SSkRbob42ZS16rDJetYCfpCkC4T6fqFY
From (redirected): https://drive.google.com/uc?id=1SSkRbob42ZS16rDJetYCfpCkC4T6fqFY&confirm=t&uuid=2c286d2c-2a18-45c5-aba8-bcd7ec30c90a
To: c:\Users\diana\Documents\AI-TF\analisis\notebooks\enaho2025\modulo_03_educacion.csv
100%|██████████| 111M/111M [00:12<00:00, 8.68MB/s] 
Downloading...
From (original): https://drive.google.com/uc?id=1snInGh

Archivos descargados satisfactoriamente!



Download completed


Cargar los 8 modulos

In [14]:
import os
import pandas as pd

carpeta_datos = "enaho2025"

modulos_info = {
    'mod_01': 'modulo_01_vivienda.csv',
    'mod_02': 'modulo_02_miembros.csv',
    'mod_03': 'modulo_03_educacion.csv',
    'mod_05': 'modulo_05_empleo.csv',
    'mod_11': 'modulo_11_servicios.csv',
    'mod_16': 'modulo_16_equipamiento.csv',
    'mod_34': 'modulo_34_sumarias.csv',
    'mod_37': 'modulo_37_programas.csv'
}

print("Cargando modulos...")
print("")

modulos = {}

for nombre_corto, nombre_archivo in modulos_info.items():
    ruta_archivo = os.path.join(carpeta_datos, nombre_archivo)
    
    if not os.path.exists(ruta_archivo):
        print(f"{nombre_corto}: NO ENCONTRADO - {ruta_archivo}")
        continue
    
    # Probar diferentes delimitadores
    delimitadores = [',', ';', '|', '\t']
    cargo_exitoso = False
    
    for delim in delimitadores:
        try:
            # Leer todo el archivo con este delimitador
            df_temp = pd.read_csv(
                ruta_archivo, 
                encoding='latin-1', 
                delimiter=delim,
                on_bad_lines='skip'
            )
            
            # Si tiene mas de 1 columna y las filas son razonables, es el correcto
            if df_temp.shape[1] > 1 and df_temp.shape[0] > 100:
                modulos[nombre_corto] = df_temp
                n_filas = len(modulos[nombre_corto])
                n_cols = modulos[nombre_corto].shape[1]
                print(f"{nombre_corto}: {n_filas:>8,} filas x {n_cols:>3} columnas [delim: '{delim}']")
                cargo_exitoso = True
                break
        except:
            continue
    
    if not cargo_exitoso:
        # Intentar con deteccion automatica
        try:
            df_temp = pd.read_csv(
                ruta_archivo, 
                encoding='latin-1', 
                sep=None, 
                engine='python',
                on_bad_lines='skip'
            )
            if df_temp.shape[1] > 1:
                modulos[nombre_corto] = df_temp
                n_filas = len(modulos[nombre_corto])
                n_cols = modulos[nombre_corto].shape[1]
                print(f"{nombre_corto}: {n_filas:>8,} filas x {n_cols:>3} columnas [delim: auto]")
                cargo_exitoso = True
        except:
            pass
    
    if not cargo_exitoso:
        print(f"{nombre_corto}: ERROR - No se pudo leer correctamente")

print("")
print(f"Total modulos cargados: {len(modulos)}/{len(modulos_info)}")

Cargando modulos...

mod_01:   44,599 filas x 336 columnas [delim: ';']
mod_02:  115,145 filas x  40 columnas [delim: ';']
mod_03:  104,446 filas x 495 columnas [delim: ';']
mod_05:   84,853 filas x 1413 columnas [delim: ';']
mod_11:  269,616 filas x  38 columnas [delim: ';']
mod_16:  202,212 filas x  47 columnas [delim: ';']
mod_34:   33,702 filas x 161 columnas [delim: ';']
mod_37:   33,702 filas x  44 columnas [delim: ';']

Total modulos cargados: 8/8


PASO 3: CONOCIENDO CADA MODULO

¿Cuántos datos? ¿Qué columnas? ¿Qué tipos?

In [26]:
print("ESTRUCTURA DE CADA MODULO")

for nombre, df in modulos.items():
    print(f"\nModulo > {nombre.upper()}")
    print(f"Filas: {df.shape[0]:,}")
    print(f"Columnas: {df.shape[1]}")
    print(f"Memoria: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
    
    # Mostrar tipos de datos resumidos
    tipos = df.dtypes.value_counts()
    tipos_str = ", ".join([f"{tipo}: {count}" for tipo, count in tipos.items()])
    print(f"Tipos: {tipos_str}")
    
    # Mostrar primeras 5 columnas (sin importar cuántas haya)
    print(f"Primeras 5 columnas: {df.columns[:5].tolist()}")
    
    # Mostrar últimas 5 columnas
    if df.shape[1] > 5:
        print(f"Ultimas 5 columnas: {df.columns[-5:].tolist()}")

print("\nEstructura revisada")

ESTRUCTURA DE CADA MODULO

Modulo > MOD_01
Filas: 44,599
Columnas: 336
Memoria: 765.3 MB
Tipos: str: 304, int64: 32
Primeras 5 columnas: ['AÑO', 'MES', 'CONGLOME', 'VIVIENDA', 'HOGAR']
Ultimas 5 columnas: ['NCONGLOME', 'SUB_CONGLOME', 'LONGITUD', 'LATITUD', 'ALTITUD']

Modulo > MOD_02
Filas: 115,145
Columnas: 40
Memoria: 179.7 MB
Tipos: str: 26, int64: 14
Primeras 5 columnas: ['AÑO', 'MES', 'CONGLOME', 'VIVIENDA', 'HOGAR']
Ultimas 5 columnas: ['T211', 'TICUEST01', 'FACPOB07', 'NCONGLOME', 'SUB_CONGLOME']

Modulo > MOD_03
Filas: 104,446
Columnas: 495
Memoria: 2758.3 MB
Tipos: str: 471, int64: 21, object: 3
Primeras 5 columnas: ['AÑO', 'MES', 'CONGLOME', 'VIVIENDA', 'HOGAR']
Ultimas 5 columnas: ['I315B', 'FACTOR07', 'FACTORA07', 'NCONGLOME', 'SUB_CONGLOME']

Modulo > MOD_05
Filas: 84,853
Columnas: 1413
Memoria: 6087.2 MB
Tipos: str: 1238, int64: 125, object: 50
Primeras 5 columnas: ['AÑO', 'MES', 'CONGLOME', 'VIVIENDA', 'HOGAR']
Ultimas 5 columnas: ['I541A', 'OCU500', 'FAC500A', 'NCONGLO

¿Qué valores tienen las primeras filas?

In [27]:
print("VISTA PREVIA DE LOS DATOS")

for nombre, df in modulos.items():
    print(f"\n{nombre.upper()} - Primeras 3 filas:")
    # Mostrar solo primeras 6 columnas para no saturar
    print(df.iloc[:, :6].head(3))
    
    print(f"\n{nombre.upper()} - Muestra aleatoria (3 filas):")
    print(df.iloc[:, :6].sample(3))
    
    print("-" * 50)

VISTA PREVIA DE LOS DATOS

MOD_01 - Primeras 3 filas:
    AÑO  MES  CONGLOME  VIVIENDA  HOGAR  UBIGEO
0  2025    1     15009        12     11   10101
1  2025    1     15009        25     11   10101
2  2025    1     15009        36     11   10101

MOD_01 - Muestra aleatoria (3 filas):
        AÑO  MES  CONGLOME  VIVIENDA  HOGAR  UBIGEO
30597  2025    9     16276        77     11   70106
31673  2025    9     17827       133     11  150132
9518   2025    3     18074       227     11  150101
--------------------------------------------------

MOD_02 - Primeras 3 filas:
    AÑO  MES  CONGLOME  VIVIENDA  HOGAR  CODPERSO
0  2025    1     15009        12     11         1
1  2025    1     15009        12     11         2
2  2025    1     15009        25     11         1

MOD_02 - Muestra aleatoria (3 filas):
        AÑO  MES  CONGLOME  VIVIENDA  HOGAR  CODPERSO
35407  2025    4     19056        22     11         1
11972  2025    2     16464        52     11         1
68390  2025    8     15706 

¿Hay celdas vacías? ¿Cuántas? ¿En qué columnas?

In [ ]:
print("ANALISIS DE VALORES NULOS")

resumen_nulos = []

for nombre, df in modulos.items():
    total_celdas = df.shape[0] * df.shape[1]
    total_nulos = df.isnull().sum().sum()
    pct_nulos = (total_nulos / total_celdas) * 100
    
    # Columnas con mas nulos
    nulos_por_col = df.isnull().sum()
    cols_con_nulos = nulos_por_col[nulos_por_col > 0]
    
    resumen_nulos.append({
        'Modulo': nombre.upper(),
        'Total Celdas': total_celdas,
        'Total Nulos': total_nulos,
        '% Nulos': f"{pct_nulos:.2f}%",
        'Cols con Nulos': len(cols_con_nulos)
    })
    
    print(f"\nModulo: {nombre.upper()}:")
    print(f"   Total celdas: {total_celdas:,}")
    print(f"   Total nulos: {total_nulos:,} ({pct_nulos:.2f}%)")
    print(f"   Columnas con nulos: {len(cols_con_nulos)} de {df.shape[1]}")
    
    # Mostrar top 3 columnas con mas nulos
    if len(cols_con_nulos) > 0:
        print(f"   Top 3 columnas con mas nulos:")
        for col, nulos in nulos_por_col.nlargest(3).items():
            pct = (nulos / len(df)) * 100
            print(f"      - {col}: {nulos:,} ({pct:.1f}%)")

# Tabla resumen
print("\n" + "-"*60)
print("RESUMEN DE VALORES NULOS:")
print("-"*60)
print(pd.DataFrame(resumen_nulos).to_string(index=False))

print("BUSCANDO VALORES QUE PARECEN NULOS CODIFICADOS")
print("="*70)

for nombre, df in modulos.items():
    print(f"\nModulo: {nombre.upper()}")
    
    # Buscar valores comunes de nulos codificados
    valores_sospechosos = [99, 999, 9999, -1, -9, 888, 99999]
    
    for col in df.columns[:10]:  # Revisar primeras 10 columnas
        for valor in valores_sospechosos:
            if valor in df[col].values:
                print(f"   Columna {col}: tiene el valor {valor} ({df[col].eq(valor).sum()} veces)")
                break

ANALISIS DE VALORES NULOS

Modulo: MOD_01:
   Total celdas: 14,985,264
   Total nulos: 0 (0.00%)
   Columnas con nulos: 0 de 336

Modulo: MOD_02:
   Total celdas: 4,605,800
   Total nulos: 0 (0.00%)
   Columnas con nulos: 0 de 40

Modulo: MOD_03:
   Total celdas: 51,700,770
   Total nulos: 0 (0.00%)
   Columnas con nulos: 0 de 495

Modulo: MOD_05:
   Total celdas: 119,897,289
   Total nulos: 0 (0.00%)
   Columnas con nulos: 0 de 1413

Modulo: MOD_11:
   Total celdas: 10,245,408
   Total nulos: 0 (0.00%)
   Columnas con nulos: 0 de 38

Modulo: MOD_16:
   Total celdas: 9,503,964
   Total nulos: 0 (0.00%)
   Columnas con nulos: 0 de 47

Modulo: MOD_34:
   Total celdas: 5,426,022
   Total nulos: 0 (0.00%)
   Columnas con nulos: 0 de 161

Modulo: MOD_37:
   Total celdas: 1,482,888
   Total nulos: 0 (0.00%)
   Columnas con nulos: 0 de 44

------------------------------------------------------------
RESUMEN DE VALORES NULOS:
------------------------------------------------------------
Modulo 